[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/tayalmanan28/MuJoCo-Tutorial/blob/main/tutorial/01_basics.ipynb)

# Tutorial 1: MuJoCo Basics

Learn the fundamentals: loading models, `MjModel`, `MjData`, and named access.

**Prerequisites:** [Tutorial 00: What is MuJoCo?](00_what_is_mujoco.ipynb) — you should already understand:
- The Describe → Simulate → Observe pattern
- `MjModel` = blueprint (constant), `MjData` = state (changes)
- `qpos`/`qvel` = positions/velocities of all joints

This tutorial goes deeper into **inspecting** models and accessing data by name.

In [1]:
pip install -q mujoco mediapy

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 232.8/232.8 kB 8.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 30.2/30.2 MB 33.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 248.5/248.5 kB 9.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.9/4.9 MB 44.2 MB/s eta 0:00:00


In [2]:
import mujoco
import numpy as np
import mediapy as media

print(f"MuJoCo version: {mujoco.__version__}")

MuJoCo version: 3.15.0


## Loading a Model

MuJoCo models are defined in [MJCF](https://mujoco.readthedocs.io/en/stable/XMLreference.html) (XML format).
The smallest valid model is `<mujoco/>`. Let's define a simple scene:

In [3]:
xml = """
<mujoco>
  <worldbody>
    <geom name="red_box" type="box" size=".2 .2 .2" rgba="1 0 0 1"/>
    <geom name="green_sphere" pos=".2 .2 .2" size=".1" rgba="0 1 0 1"/>
  </worldbody>
</mujoco>
"""
model = mujoco.MjModel.from_xml_string(xml)
print(f"Number of geoms: {model.ngeom}")

Number of geoms: 2


## MjModel — The Static Description

`MjModel` contains everything that **doesn't change** during simulation:
geometry sizes, masses, joint types, colors, etc.

In [4]:
# Number of geoms and their RGBA colors
print(f"ngeom: {model.ngeom}")
print(f"geom_rgba:\n{model.geom_rgba}")

ngeom: 2
geom_rgba:
[[1. 0. 0. 1.]
 [0. 1. 0. 1.]]


## Named Access

The Python bindings provide convenient named accessors:

In [7]:
# Access by name
print("green_sphere rgba:", model.geom('green_sphere').rgba)
print("green_sphere id:", model.geom('green_sphere').id)
print("geom 0 name:", model.geom(0).name)

# List all geom names
print("All geoms:", [model.geom(i).name for i in range(model.ngeom)])

green_sphere rgba: [0. 1. 0. 1.]
green_sphere id: 1
geom 0 name: red_box
All geoms: ['red_box', 'green_sphere']


## MjData — The Dynamic State

`MjData` contains the simulation **state** (time, positions, velocities)
and derived quantities (Cartesian positions, forces, etc.).

In [8]:
data = mujoco.MjData(model)

# Derived quantities need explicit computation
print("Before mj_kinematics:", data.geom_xpos)

mujoco.mj_kinematics(model, data)
print("After mj_kinematics:", data.geom_xpos)
print("green_sphere pos:", data.geom('green_sphere').xpos)

Before mj_kinematics: [[0. 0. 0.]
 [0. 0. 0.]]
After mj_kinematics: [[0.  0.  0. ]
 [0.2 0.2 0.2]]
green_sphere pos: [0.2 0.2 0.2]


## Key Takeaways

- `MjModel` = static description (compiled from XML)
- `MjData` = dynamic state + derived quantities
- Use `mj_forward()` or `mj_kinematics()` to propagate state before reading derived quantities
- Named access: `model.geom('name')`, `data.body('name')`, etc.